In [79]:
#Final cleaned version of English BTC-related tweets. Includes only relevant, non-spam, non-duplicated content with proper formatting and metadata.

In [80]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tnrange, tqdm_notebook, tqdm

In [ ]:
selected_columns = ['date', 'n_replies', 'n_likes', 'n_retweets', 'text']
tweets_clean_file = pd.read_csv('engtweetsbtc_clean.csv', sep=';', usecols=selected_columns)

print("Data shape:", tweets_clean_file.shape)
print("\nColumn names:")
tweets_clean_file.columns.tolist()
tweets_clean_file.head(3)

In [ ]:
# Ensure n_retweets is numeric
tweets_clean_file['n_retweets'] = pd.to_numeric(tweets_clean_file['n_retweets'], errors='coerce')

# Filter out rows where n_retweets is 0 or NaN
tweets_clean_file_filtered = tweets_clean_file[tweets_clean_file['n_retweets'] > 0].copy()
tweets_clean_file_filtered.head(3)
print(f"Filtered: {len(tweets_clean_file_filtered)} tweets with retweets > 0")

In [ ]:
# First, check date formats
sample_dates = tweets_clean_file_filtered['date'].head(20).tolist()
for i, date_str in enumerate(sample_dates):
    print(f"Row {i}: '{date_str}'")

# Clean date strings first
def clean_date_string(date_str):
    if pd.isna(date_str):
        return date_str
    # Remove timezone part if present
    if isinstance(date_str, str):
        if '+' in date_str:
            return date_str.split('+')[0].strip()
    return date_str

tweets_clean_file_filtered['date_clean'] = tweets_clean_file_filtered['date'].apply(clean_date_string)

# Now convert to datetime
tweets_clean_file_filtered['date_parsed'] = pd.to_datetime(tweets_clean_file_filtered['date_clean'], errors='coerce', format='mixed')

# Remove rows with invalid dates
invalid_dates = tweets_clean_file_filtered['date_parsed'].isna().sum()
print(f"\nInvalid dates found: {invalid_dates}")
tweets_clean_file_filtered = tweets_clean_file_filtered[tweets_clean_file_filtered['date_parsed'].notna()].copy()

# Filter by date range
start_date = pd.to_datetime('2019-03-30')
end_date = pd.to_datetime('2022-07-28')
df_final = tweets_clean_file_filtered[(tweets_clean_file_filtered['date_parsed'] >= start_date) & (tweets_clean_file_filtered['date_parsed'] <= end_date)]

# Keep only needed columns
df_final = df_final[['date_parsed', 'n_replies', 'n_likes', 'n_retweets', 'text']]
df_final = df_final.rename(columns={'date_parsed': 'date'})

print(f"\nFinal dataset: {len(df_final)} rows")
print(df_final.head())
df_final.to_csv('filtered_tweets_final.csv', sep=';', index=False, encoding='utf-8')
